# 3D化粧ロボット
AI生成した4人の顔から、鼻・頬・顎のある顔前面の3Dメッシュを推定し、ブラシを曲面へ合わせます。

**旧ノートブックとは別に、上から実行。顔画像は「1. コードと画像を配置」でアップロード。GPU不要。**
全体実行は未確認なので、短い確認から始めてください。接触力・皮膚変形は再現しません。

In [ ]:
from pathlib import Path
import sys, subprocess, os
project=Path('/content/makeup_robot_3d');project.mkdir(exist_ok=True)
env=Path('/content/makeup_robot_3d_env')
def logged(cmd):
    with subprocess.Popen([str(x) for x in cmd],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1','MPLBACKEND':'Agg'}) as proc:
        for line in proc.stdout: print(line,end='',flush=True)
        rc=proc.wait()
    if rc: raise RuntimeError(f'終了コード {rc}。直前のエラー表示を共有してください。')
logged([sys.executable,'-m','pip','install','-q','uv'])
if not (env/'bin/python').exists():
    logged([sys.executable,'-m','uv','venv','--python','3.11',env])
PY=env/'bin/python'
logged([sys.executable,'-m','uv','pip','install','--python',PY,'--only-binary',':all:',
'numpy==1.26.4','scipy==1.15.3','Pillow==11.3.0','mediapipe==0.10.21','opencv-contrib-python==4.11.0.86','pybullet==3.2.7','imageio==2.37.0','imageio-ffmpeg==0.6.0'])
print('Python 3.11の専用環境を用意しました。')

## 1. コードと画像を配置
セルを実行するとファイル選択が出ます。ChatGPTで生成した「4人の女性の正面顔リファレンス.png」を選んでください。画像の埋め込みを外した軽量版です。

In [ ]:
from pathlib import Path
project = Path('/content/makeup_robot_3d')
(project/'makeup_3d.py').write_text('"""AI-generated portraits -> estimated 3D face surface -> robot makeup.\nGeometry is a monocular estimate, not measured anatomy. No force control.\n"""\nfrom pathlib import Path\nimport argparse, json, urllib.request, os\nos.environ["MPLBACKEND"] = "Agg"\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFilter\nfrom scipy.spatial import Delaunay, cKDTree\n\nROOT=Path(__file__).resolve().parent\nOUT=ROOT/\'results\'; OUT.mkdir(exist_ok=True)\nMODEL_URL=\'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task\'\nROSE=np.array([196,104,126],dtype=float) # Approximation from supplied photograph, not a shade match.\nTOOL=.055\n\nclass FaceSurface:\n    def __init__(self, points, position):\n        self.uv=np.asarray(points)[:468,:2]\n        self.depth=np.asarray(points)[:468,2]\n        self.center=np.mean(self.uv[[234,454]],axis=0)\n        self.scale=.16/np.linalg.norm(self.uv[234]-self.uv[454])\n        self.position=np.asarray(position,float)\n        # x = image vertical; y = image horizontal; z = negative estimated depth.\n        self.vertices=np.column_stack(((self.uv[:,1]-self.center[1])*self.scale,\n                         (self.uv[:,0]-self.center[0])*self.scale,\n                         -(self.depth-np.median(self.depth))*self.scale))+self.position\n        self.tri=Delaunay(self.uv)\n        self.tree=cKDTree(self.vertices)\n        self.cheeks=self.uv[[50,280]].copy()\n    def sample(self,uv):\n        uv=np.asarray(uv,float)\n        index=int(self.tri.find_simplex(uv))\n        if index<0: raise ValueError(\'Brush path outside face mesh; stopping.\')\n        transform=self.tri.transform[index]\n        bary=transform[:2]@(uv-transform[2]);weights=np.r_[bary,1-bary.sum()]\n        ids=self.tri.simplices[index];a,b,c=self.vertices[ids]\n        normal=np.cross(b-a,c-a);normal/=np.linalg.norm(normal)\n        if normal[2]<0:normal=-normal\n        return weights@self.vertices[ids],normal\n    def export_obj(self,path):\n        # OBJ triangle winding must face upwards in world space.\n        lines=[\'# AI estimated anterior face surface; world units metres\']\n        lines += [\'v \'+\' \'.join(map(str,p)) for p in self.vertices]\n        lines += [f\'vt {u} {1-v}\' for u,v in self.uv]\n        for ids in self.tri.simplices:\n            a,b,c=self.vertices[ids]\n            if np.cross(b-a,c-a)[2]<0: ids=ids[::-1]\n            lines.append(\'f \'+\' \'.join(f\'{i+1}/{i+1}\' for i in ids))\n        path.write_text(\'\\n\'.join(lines))\n    def nearest_uv(self,point):\n        # Approximate projection is refined by planar barycentric interpolation.\n        u=(point[1]-self.position[1])/self.scale+self.center[0]\n        v=(point[0]-self.position[0])/self.scale+self.center[1]\n        return np.array([u,v])\n\ndef quaternion_z(axis):\n    z=np.array([0.,0.,1.]);axis=np.asarray(axis);axis=axis/np.linalg.norm(axis)\n    dot=float(z@axis)\n    if dot<-.999999:return [1.,0.,0.,0.]\n    q=np.r_[np.cross(z,axis),1+dot];return (q/np.linalg.norm(q)).tolist()\n\ndef detect(images):\n    import mediapipe as mp\n    from mediapipe.tasks import python\n    from mediapipe.tasks.python import vision\n    path=ROOT/\'face_landmarker.task\'\n    if not path.exists():urllib.request.urlretrieve(MODEL_URL,path)\n    options=vision.FaceLandmarkerOptions(base_options=python.BaseOptions(model_asset_path=str(path)),num_faces=1,\n            min_face_detection_confidence=.5,min_face_presence_confidence=.5)\n    points=[]\n    with vision.FaceLandmarker.create_from_options(options) as detector:\n        for i,im in enumerate(images):\n            result=detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=np.asarray(im,dtype=np.uint8).copy()))\n            if len(result.face_landmarks)!=1:raise RuntimeError(f\'Face {i+1} not detected. Use a frontal adult portrait.\')\n            points.append(np.array([[p.x,p.y,p.z] for p in result.face_landmarks[0]]))\n    return points\n\ndef make_compact_texture():\n    im=Image.new(\'RGB\',(512,320),(25,25,28));d=ImageDraw.Draw(im)\n    d.rounded_rectangle((15,15,497,305),radius=22,fill=(182,186,191))\n    d.rectangle((34,34,320,286),fill=tuple(ROSE.astype(int)))\n    d.rectangle((345,34,478,286),fill=(18,18,20))\n    d.text((110,118),\'Dior\',fill=(168,80,102),font_size=50)\n    d.text((350,247),\'CD\',fill=(215,216,219),font_size=30)\n    im.save(OUT/\'compact_reference_style.png\');return im\n\ndef run(atlas,smoke=False):\n    import pybullet as p, pybullet_data, imageio.v2 as iio\n    from importlib.metadata import version\n    original=Image.open(atlas).convert(\'RGB\');w,h=original.size\n    images=[original.crop((col*w//2,row*h//2,(col+1)*w//2,(row+1)*h//2)).resize((512,512)) for row in range(2) for col in range(2)]\n    points=detect(images)\n    surfaces=[FaceSurface(x,[.57+dx,dy,.32]) for x,(dx,dy) in zip(points,[(0,0),(.025,.035),(-.025,-.03),(.02,-.025)])]\n    for i,(im,s) in enumerate(zip(images,surfaces)):\n        im.save(OUT/f\'portrait_{i+1}.png\');s.export_obj(OUT/f\'face_{i+1}.obj\')\n    # Baseline: same first portrait and same world-space path for all later trials.\n    reference=surfaces[0]\n    p.connect(p.DIRECT);p.setAdditionalSearchPath(pybullet_data.getDataPath())\n    p.setGravity(0,0,-9.81);p.setTimeStep(1/240)\n    p.loadURDF(\'plane.urdf\');robot=p.loadURDF(\'kuka_iiwa/model.urdf\',useFixedBase=True)\n    def box(half,pos,color):\n        return p.createMultiBody(0,-1,p.createVisualShape(p.GEOM_BOX,halfExtents=half,rgbaColor=color),basePosition=pos)\n    box([.17,.15,.015],[.57,0,.245],[.19,.21,.25,1])\n    # Compact is an illustrative black/silver case with a rose pan, not an exact CAD replica.\n    box([.065,.05,.009],[.44,-.26,.27],[.06,.06,.07,1])\n    box([.061,.046,.002],[.44,-.26,.281],[.72,.74,.77,1])\n    box([.038,.039,.002],[.424,-.26,.284],[*(ROSE/255),1])\n    box([.014,.039,.002],[.48,-.26,.284],[.04,.04,.05,1])\n    brush=p.createMultiBody(0,-1,p.createVisualShape(p.GEOM_CYLINDER,radius=.006,length=TOOL,\n                    visualFramePosition=[0,0,TOOL/2],rgbaColor=[.06,.06,.08,1]))\n    bristle=p.createMultiBody(0,-1,p.createVisualShape(p.GEOM_SPHERE,radius=.007,rgbaColor=[.77,.41,.49,1]))\n    compact=make_compact_texture().resize((220,138))\n    view=p.computeViewMatrix([1.02,-.82,.94],[.48,0,.32],[0,0,1])\n    close=p.computeViewMatrix([.82,-.30,.65],[.57,0,.32],[0,0,1])\n    proj=p.computeProjectionMatrixFOV(45,1.,.02,3)\n    frame_count=0;rows=[];body=None\n    video=OUT/(\'smoke_test.mp4\' if smoke else \'makeup_3d_demo.mp4\')\n    writer=iio.get_writer(str(video),fps=20,codec=\'libx264\',quality=8,macro_block_size=16)\n    try:\n        for index,(im,surface) in enumerate(zip(images,surfaces)):\n            if smoke and index>0:break\n            for mode in [\'FIXED\',\'AI\']:\n                if body is not None:p.removeBody(body)\n                vis=p.createVisualShape(p.GEOM_MESH,fileName=str(OUT/f\'face_{index+1}.obj\'),rgbaColor=[1,1,1,1])\n                body=p.createMultiBody(0,-1,vis)\n                texture=OUT/f\'painted_{index}_{mode}.png\';im.save(texture)\n                texid=p.loadTexture(str(texture));p.changeVisualShape(body,-1,textureUniqueId=texid)\n                for j,q in enumerate([0,.4,0,-1.5,0,1.1,0]):p.resetJointState(robot,j,q)\n                paint=np.zeros((512,512),float); yy,xx=np.mgrid[:512,:512]\n                tracking=[];reference_error=[];surface_distance=[];hits=0;attempts=0;painted=im.copy()\n                for cheek in range(2):\n                    start=np.asarray(p.getLinkState(robot,6,computeForwardKinematics=True)[4])\n                    steps=20 if smoke else 100\n                    for f in range(steps):\n                        phase=f/max(1,steps-1);draw=.20<=phase<=.85\n                        t=np.clip((phase-.20)/.65,0,1)\n                        offset_m=np.array([.009*np.cos(4*np.pi*t),.006*np.sin(4*np.pi*t)]) if draw else np.zeros(2)\n                        desired,n=surface.sample(surface.cheeks[cheek]+offset_m/surface.scale)\n                        chosen=surface if mode==\'AI\' else reference\n                        contact,normal=chosen.sample(chosen.cheeks[cheek]+offset_m/chosen.scale)\n                        lift=.003 if draw else .045\n                        goal=contact+normal*(TOOL+lift)\n                        if phase<.20:goal=start+(goal-start)*min(1,phase/.20)\n                        orn=quaternion_z(-normal)\n                        joints=p.calculateInverseKinematics(robot,6,goal.tolist(),orn,maxNumIterations=120,residualThreshold=1e-5)\n                        p.setJointMotorControlArray(robot,list(range(7)),p.POSITION_CONTROL,targetPositions=joints[:7],forces=[300]*7)\n                        for _ in range(12):p.stepSimulation()\n                        state=p.getLinkState(robot,6,computeForwardKinematics=True)\n                        actual=np.asarray(state[4]);rotation=np.asarray(p.getMatrixFromQuaternion(state[5])).reshape(3,3)\n                        tip=actual+rotation@np.array([0,0,TOOL])\n                        p.resetBasePositionAndOrientation(brush,actual.tolist(),state[5]);p.resetBasePositionAndOrientation(bristle,tip.tolist(),[0,0,0,1])\n                        if draw:\n                            attempts+=1\n                            tracking.append(float(np.linalg.norm(tip-(contact+normal*.003))*1000))\n                            reference_error.append(float(np.linalg.norm(tip-(desired+n*.003))*1000))\n                            uv=surface.nearest_uv(tip)\n                            try:\n                                projected,n2=surface.sample(uv)\n                                distance=float(np.linalg.norm(tip-projected));surface_distance.append(distance*1000)\n                                if distance<.008:\n                                    hits+=1; x,y=uv*512\n                                    paint+=.035*np.exp(-((xx-x)**2+(yy-y)**2)/(2*9**2))\n                                    alpha=np.minimum(paint,.42)[:,:,None]\n                                    painted=Image.fromarray(np.uint8(np.asarray(im)*(1-alpha)+ROSE*alpha))\n                            except ValueError:pass\n                        if f%4==0:\n                            p.changeTexture(texid,np.asarray(painted,dtype=np.uint8).ravel().tolist(),512,512)\n                        def render(v):\n                            data=p.getCameraImage(512,512,v,proj,renderer=p.ER_TINY_RENDERER)[2]\n                            if isinstance(data,(bytes,bytearray)):arr=np.frombuffer(data,dtype=np.uint8)\n                            else:arr=np.asarray(data,dtype=np.uint8)\n                            return Image.fromarray(arr.reshape(512,512,4)[:,:,:3])\n                        frame=Image.new(\'RGB\',(1280,720),(18,23,34));d=ImageDraw.Draw(frame)\n                        frame.paste(render(view),(16,88));frame.paste(render(close).resize((420,420)),(540,88))\n                        frame.paste(painted.resize((272,272)),(992,88));frame.paste(compact,(1010,405))\n                        d.text((24,18),\'3D MAKEUP ROBOT\',fill=\'white\',font_size=30)\n                        d.text((24,57),f\'Portrait {index+1}/4 | {mode} | cheek {cheek+1}/2\',fill=(237,166,185),font_size=20)\n                        d.text((540,530),\'Estimated curved face surface\',fill=\'white\',font_size=18)\n                        d.text((992,375),\'Generated portrait + blush\',fill=\'white\',font_size=16)\n                        d.text((992,555),\'Dior reference-style compact\',fill=(210,210,215),font_size=15)\n                        d.text((24,624),\'AI: image landmarks -> 3D mesh -> surface-normal brush path -> joint control\',fill=\'white\',font_size=19)\n                        d.text((24,656),\'Monocular depth estimate | assumed face width 160 mm | no contact-force simulation\',fill=(185,192,208),font_size=17)\n                        d.text((24,687),\'Pink paint is deposited from the actual simulated brush tip near the surface.\',fill=(185,192,208),font_size=16)\n                        writer.append_data(np.asarray(frame));frame_count+=1\n                    print(f\'Portrait {index+1}/4 {mode} cheek {cheek+1} done\',flush=True)\n                row={\'portrait\':index+1,\'mode\':mode,\'tracking_error_mm\':float(np.mean(tracking)),\n                     \'ai_reference_path_error_mm\':float(np.mean(reference_error)),\n                     \'near_surface_frames\':hits,\'drawing_frames\':attempts,\n                     \'near_surface_fraction\':hits/max(1,attempts)}\n                rows.append(row);print(json.dumps(row),flush=True)\n                frame.save(OUT/f\'preview_{index+1}_{mode}.png\')\n    finally:writer.close();p.disconnect()\n    result={\'status\':\'smoke_test\' if smoke else \'completed\',\'video_seconds\':frame_count/20,\'fps\':20,\n        \'depth\':\'MediaPipe monocular relative depth; assumed face width 0.16m\',\n        \'metrics_note\':\'AI reference is model-derived, NOT independently measured ground truth. No face-detection accuracy claim.\',\n        \'product_note\':\'Appearance approximation based on user-supplied Dior photo; exact shade unknown.\',\n        \'versions\':{x:version(x) for x in [\'mediapipe\',\'pybullet\',\'numpy\',\'Pillow\']},\'trials\':rows}\n    (OUT/(\'smoke_results.json\' if smoke else \'results.json\')).write_text(json.dumps(result,indent=2))\n    if not smoke:\n        report=\'\'\'# AI生成顔の3D表面推定を用いたチーク塗布ロボット\\n\\n## 目的・何をしたか\\n架空の成人4人の顔画像から顔の3次元表面を推定し、ロボットのブラシが左右の頬に沿って動くシミュレーションを作成した。顔ごとに位置を調整する方式と、最初の顔の軌道を固定して使う方式を比較した。\\n\\n## 方法\\n事前にAI生成した4枚の画像を使用。試行中に新規画像生成はしない。MediaPipe Face Landmarkerで顔特徴点を検出し、相対奥行きと顔幅160mmの仮定を用いて前面メッシュを作成した。頬周辺の経路を三角形上に補間し、面の法線に合わせてブラシ姿勢を指定。PyBulletの逆運動学と位置制御でKUKA iiwaを動かす。実際のブラシ先端が表面の8mm以内に来た場合にだけ色を重ねた。\\n\\n## 結果\\n以下は今回の実行ログであり、知覚精度や実機の安全性を示す値ではない。AI参照軌道自体が同じモデルから得られた推定値で、独立した正解ではない。\\n\\n\'\'\'\n        for r in rows:report+=f"- 顔{r[\'portrait\']} / {r[\'mode\']}: 追従誤差 {r[\'tracking_error_mm\']:.2f}mm、AI参照軌道との差 {r[\'ai_reference_path_error_mm\']:.2f}mm、表面近傍フレーム率 {r[\'near_surface_fraction\']:.1%}\\n"\n        report+=\'\'\'\\n## 工夫\\n同一顔について固定方式とAI方式を比較し、同じ制御系・塗布条件を使用した。ブラシの目標位置だけで塗布を描かず、物理ステップ後の実際の手先位置からブラシ先端を算出した。チークは提供されたDiorの商品写真を参考にローズ系色と黒・銀のケースを表現した。正確な商品CAD・色番号の再現ではない。\\n\\n## 限界・今後\\n顔前面のみの推定メッシュであり、頭蓋全体の復元や精密スキャンではない。顔モデル・ブラシは可視化用で、顔との接触力、皮膚変形、圧力制御、障害物回避の検証はしていない。曲面追従の概念実証として評価する。今後は独立した3D正解データで評価し、力覚センサーを用いた接触制御を追加する。前の合成図形版で得られた6.45mm等の数値は本実験には流用しない。\\n\\n## 出典\\nhttps://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker/python\\nhttps://github.com/bulletphysics/bullet3\\n画像生成・コード作成支援: ChatGPT。\\n\'\'\'\n        (OUT/\'report_draft.md\').write_text(report,encoding=\'utf-8\')\n    print(\'DONE: \'+str(video),flush=True)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--atlas\',required=True);parser.add_argument(\'--smoke\',action=\'store_true\')\n    args=parser.parse_args();run(args.atlas,args.smoke)\n', encoding='utf-8')
(project/'README.md').write_text('# 化粧ロボット 3D版\n\n## 始め方\nMakeup_Robot_3D_Colab.ipynb を https://colab.research.google.com/ にアップロードし、上から実行してください。旧ノートブックとは別に開きます。コードは埋め込み済みです。画像の選択時には、このリポジトリの portraits.png をアップロードしてください。GPUは不要。Python 3.11の専用環境を自動作成します。Colab本体のPythonは変更しません。\n\nまず4秒の確認動画を生成し、次に本番80秒（4人×2方式×2頬×5秒）を生成します。1頬ごとにログが出ます。\n\n## 変更内容\n事前にAI生成した架空の成人4人の顔を使用します。試行中に新しく画像を生成するAPI連携は含みません。MediaPipeで顔特徴点と相対奥行きを推定し、顔幅160mmの仮定で顔前面メッシュを生成。ブラシの向きを曲面の法線へ合わせ、実際の先端が表面8mm以内に来た時のみ色を重ねます。参考画像に寄せたローズ色と黒・銀のコンパクトを配置します。正確なDiorの商品CADや色番号は再現していません。\n\n## 検証状況\nPython構文と、合成メッシュ100点で座標投影・法線・ブラシ姿勢を確認済み。MediaPipeとPyBulletを含む全体実行は未確認です。短い確認セルから実行してください。\n\n## 限界\nこれは顔前面の推定3D表面で、頭全体の精密モデルではありません。顔とブラシは可視化用で接触力を計算しません。皮膚変形・圧力制御・衝突回避は未実装。曲面に沿ったロボット動作の概念実証です。\n\n## 評価\n固定方式は最初の顔に合わせた軌道を使い、AI方式は各顔に合わせます。各顔・位置について両方式を試します。独立した正解形状がないので、検出精度は報告しません。記録するのは追従誤差、AI参照軌道との差、表面近傍フレーム率です。最初の顔は両方式が同条件です。旧平面版の6.45mm・83.5%は今回の結果に流用しないでください。\n\n## 生成物\nresults/makeup_3d_demo.mp4、results/results.json、results/report_draft.md、4つのOBJモデルと画像。レポートは原稿で、PDF化は動作・数値確認後に行います。課題提出は別途必要です。\n\n## 参考\nhttps://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker/python\nhttps://github.com/bulletphysics/bullet3\n画像生成・コード作成支援: ChatGPT。\n\n## 動画\nhttps://youtu.be/VlaPdqWlWuc\n\n## 実行環境の修正\nColabから引き継ぐMatplotlibの描画設定をAggに固定し、専用環境でのimportエラーを回避します。結果表示セルでもprojectを定義しています。\n\n## ローカル実行（Python 3.11）\n```bash\npython -m pip install -r requirements.txt\npython makeup_3d.py --atlas portraits.png --smoke\npython makeup_3d.py --atlas portraits.png\n```\n初回はMediaPipeモデルをGoogleの配布元からダウンロードします。結果はresults/に保存されます。\n', encoding='utf-8')
(project/'requirements.txt').write_text('numpy==1.26.4\nscipy==1.15.3\nPillow==11.3.0\nmediapipe==0.10.21\nopencv-contrib-python==4.11.0.86\npybullet==3.2.7\nimageio==2.37.0\nimageio-ffmpeg==0.6.0\n',encoding="utf-8")
from google.colab import files
print("4人の顔が並んだ画像を選んでください。")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("顔画像を1枚だけ選んでください。")
(project / "portraits.png").write_bytes(next(iter(uploaded.values())))
print("コードと顔画像の配置完了")

## 2. 短い動作確認（4秒）
最初の顔だけで実行します。短縮版なので動きは急になります。本番では各頬に5秒かけます。顔・アームが表示されるか確認してください。

In [ ]:
logged([PY,project/'makeup_3d.py','--atlas',project/'portraits.png','--smoke'])
from IPython.display import Video,display
display(Video(str(project/'results/smoke_test.mp4'),embed=True))

## 3. 本番動画を作る（80秒）
短い確認が通ったら実行してください。処理は数分以上かかる場合があります。ログは1頬ごとに更新されます。

In [ ]:
logged([PY,project/'makeup_3d.py','--atlas',project/'portraits.png'])
display(Video(str(project/'results/makeup_3d_demo.mp4'),embed=True))

## 4. 結果とレポート原稿
独立した正解形状はありません。参照軌道との差は、知覚精度や安全性の証明にはなりません。

In [ ]:
from pathlib import Path
project = Path('/content/makeup_robot_3d')
import json
from IPython.display import Markdown
print(json.dumps(json.loads((project/'results/results.json').read_text()),ensure_ascii=False,indent=2))
display(Markdown((project/'results/report_draft.md').read_text()))

## 5. 生成物をダウンロード
動画・モデル・結果・コード・レポート原稿を保存。PDF化と提出は、結果を確認してから行います。

In [ ]:
import zipfile
from google.colab import files
archive=Path('/content/makeup_robot_3d_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for file in project.rglob('*'):
        if file.is_file() and '__pycache__' not in str(file): z.write(file,file.relative_to(project))
files.download(str(archive))